# Interval-sample variability

Summarize repeated interval samples by channel and in 30-second windows. Invalid samples break adjacent differences. Unique NN beats are unverified. Spectra use recorded elapsed time and do not support clinical LF/HF interpretation. Question summaries require a documented acquisition timezone and clock synchronization confirmed by acquisition records. Neither is available here, so these summaries are skipped.

Use `MMS_DATA_ROOT` for another dataset and `MMS_OUTPUT_ROOT` for exports outside source data.


In [ ]:
import matplotlib.pyplot as plt
from mms import exploration as ex, hrv

SOURCE = "case-study"

intervals = ex.recordings("ibi", source=SOURCE)
interval_table = ex.interval_summary(intervals)
print(interval_table)
ex.plot_traces(intervals, "ibi")
features = ex.aligned_questions(source=SOURCE)
if not features.empty:
    metrics = ["mean_hr_bpm", "interval_rmsdiff_ms", "mean_pupil_source_units"]
    print(features.groupby(["Session", "Type", "Channel"])[metrics].mean())


In [ ]:
ex.plot_summary(interval_table, "Sample SD (ms)")
metrics = ["Sample SD (ms)", "Adjacent-sample RMS difference (ms)"]
print(ex.baseline_changes(interval_table, metrics))

for label, frame in intervals.items():
    windows = hrv.hrv_over_time(frame, window_s=30, step_s=30)
    columns = ["iSensor", "n_samples", "n_pairs", "sdnn", "rmssd"]
    print(label, "30-second windows:", windows[columns].describe())


In [ ]:
spectra = ex.interval_spectra(intervals)
n_spectra = max(1, len(spectra))
fig, axes = plt.subplots(
    n_spectra, 1, figsize=(9, 2.5 * n_spectra), squeeze=False
)
for ax, (label, spectrum) in zip(axes.flat, spectra.items()):
    ax.plot(spectrum["Frequency (Hz)"], spectrum["Normalized sample power"])
    ax.set_title(label)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Normalized sample power")
fig.suptitle("Recorded interval-sample spectra; no NN-beat or autonomic interpretation")
fig.tight_layout()
plt.show()
plt.close(fig)
